# 05 · Trip-cost model — training and evaluation
Gradient boosting on daily cost with quantile bands (`ml/cost/train_cost_model.py`). Training data = synthetic taste-driven journals **plus** real-sourced entries whose costs come from observed prices.

In [1]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
ROOT = r"C:\xampp\htdocs\travel_journel"
os.chdir(ROOT); sys.path.insert(0, ROOT)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 70)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": .3})
DATA = os.path.join(ROOT, "ml", "data")

In [2]:
from ml.cost.train_cost_model import load_training_frame
df = load_training_frame(); print(df.shape)
df.head(3)

(2881, 16)


,entry_id,dest,style,duration_days,dv,true_total,food_total,transport_total,accommodation_total,shopping_total,fees_misc_total,region,popularity_tier,base_daily,season,party_size
0,2022,"Mathura, India",luxury,3,2026-07-15,43000.0,4500.0,30000.0,8500.0,0.0,0.0,North India,mainstream,2750.0,monsoon,1
1,9282,"Raipur, India",mid-range,5,2025-12-02,7750.0,1700.0,1470.0,3020.0,850.0,710.0,Central India,mainstream,1300.0,winter,1
2,9283,"Manjeshwar, India",backpacker,5,2024-02-14,7250.0,1650.0,1300.0,2840.0,940.0,520.0,South India,mainstream,1350.0,winter,1


In [3]:
import subprocess
out = subprocess.run([sys.executable, "-m", "ml.cost.train_cost_model"], capture_output=True, text=True, cwd=ROOT)
print(out.stdout[-2500:], out.stderr[-500:])

Training rows: 2881

=== Cost model metrics (held-out 20%) ===
MAE : Rs 2,736
R2  : 0.813
MAPE: 16.0%

Sanity (Mathura, India, 5 days): backpacker=Rs14268 < luxury=Rs15845
artifact -> cost_model.pkl
 


In [4]:
from ml.cost.predict import model_metrics, predict_cost
print(model_metrics())
import scrapbook as sb; sb.glue("cost_metrics", model_metrics())

{'mae': 2735.8525235383645, 'r2': 0.8126719460779792, 'mape': 16.04931308229351}


In [5]:
rows=[]
for d in ["Goa, India","Jaipur, India","Leh, India","Kochi, India","Udaipur, India"]:
    try:
        p = predict_cost(d, duration_days=5, travel_style="mid-range", month=12)
        rows.append({"destination":d, **{k:v for k,v in p.items() if not isinstance(v,(dict,list))}})
    except Exception as e: rows.append({"destination":d,"error":str(e)[:60]})
pd.DataFrame(rows)

,destination,predicted_cost,low,high,per_day,duration_days,currency
0,"Goa, India",11516,8856,14672,2303,5,INR
1,"Jaipur, India",10800,8467,13988,2160,5,INR
2,"Leh, India",5626,5371,7651,1125,5,INR
3,"Kochi, India",11106,8447,14379,2221,5,INR
4,"Udaipur, India",13024,9223,15951,2605,5,INR
